In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier


# ============================================================
# 【读取步骤】
# 读取最终清洗后的训练集和测试集
# ============================================================

model_train = pd.read_parquet(
    "data/processed/model_train_clean.parquet"
)

model_test = pd.read_parquet(
    "data/processed/model_test_clean.parquet"
)


# 1. 保存测试集客户编号
test_customer_ids = (
    model_test["SK_ID_CURR"]
    .copy()
)


# 2. 提取目标变量
y = (
    model_train["TARGET"]
    .astype("int8")
    .copy()
)


# 3. 建立完整训练特征
X = (
    model_train
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)


# 4. 建立测试特征
X_test = (
    model_test
    .drop(
        columns=["SK_ID_CURR"]
    )
    .copy()
)


print("X Shape:", X.shape)
print("X_test Shape:", X_test.shape)
print(
    "Train/Test字段是否一致:",
    X.columns.tolist() == X_test.columns.tolist()
)

X Shape: (307511, 439)
X_test Shape: (48744, 439)
Train/Test字段是否一致: True


In [2]:
# ============================================================
# 【检查步骤】
# 识别类别字段和数值字段
# ============================================================

categorical_columns = (
    X
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_columns = (
    X
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)


print(
    "类别特征数量:",
    len(categorical_columns)
)

print(
    "数值特征数量:",
    len(numeric_columns)
)

print(
    "总特征数量:",
    len(categorical_columns)
    + len(numeric_columns)
)

类别特征数量: 24
数值特征数量: 415
总特征数量: 439


In [3]:
# ============================================================
# 【最终模型】
# 建立最终 LightGBM Pipeline
# ============================================================

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_columns
        ),
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)


final_lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


final_lightgbm_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor
        ),
        (
            "model",
            final_lightgbm_model
        )
    ]
)

In [4]:
# ============================================================
# 【模型训练】
# 使用全部训练客户训练最终模型
# ============================================================

final_lightgbm_pipeline.fit(
    X,
    y
)

print(
    "最终 LightGBM 模型训练完成"
)

最终 LightGBM 模型训练完成


In [5]:
# ============================================================
# 【特征解释】
# 提取 LightGBM 特征重要性
# ============================================================

# 1. 取得预处理后的特征名称
processed_feature_names = (
    final_lightgbm_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


# 2. 取得模型特征重要性
feature_importance_values = (
    final_lightgbm_pipeline
    .named_steps["model"]
    .feature_importances_
)


# 3. 建立特征重要性表
feature_importance = pd.DataFrame(
    {
        "feature": processed_feature_names,
        "importance": feature_importance_values
    }
)


# 4. 清理字段名前缀
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace(
        r"^(numeric|categorical)__",
        "",
        regex=True
    )
)


# 5. 按重要性降序排列
feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


feature_importance.head(30)

,feature,importance
0,EXT_SOURCE_1,505
1,EXT_SOURCE_2,450
2,EXT_SOURCE_3,440
3,AMT_ANNUITY,247
4,CREDIT_TO_GOODS,247
5,AMT_GOODS_PRICE,229
6,AGE_YEARS,223
7,DAYS_BIRTH,211
8,AMT_CREDIT,188
9,LATEST_CREDIT_DAYS,187


In [6]:
feature_importance.to_csv(
    "data/processed/lightgbm_feature_importance.csv",
    index=False
)

print(
    "特征重要性保存完成"
)

特征重要性保存完成


In [7]:
# ============================================================
# 【模型预测】
# 预测测试集客户的违约概率
# ============================================================

test_default_probability = (
    final_lightgbm_pipeline
    .predict_proba(X_test)[:, 1]
)


print(
    "预测概率最小值:",
    test_default_probability.min()
)

print(
    "预测概率最大值:",
    test_default_probability.max()
)

print(
    "预测概率平均值:",
    test_default_probability.mean()
)

预测概率最小值: 0.018304183534965275
预测概率最大值: 0.9464493004442622
预测概率平均值: 0.3618094275100992


In [8]:
# ============================================================
# 【建立结果表】
# 保存每位测试客户的违约概率
# ============================================================

test_predictions = pd.DataFrame(
    {
        "SK_ID_CURR": test_customer_ids,
        "TARGET": test_default_probability
    }
)


print(
    "预测结果 Shape:",
    test_predictions.shape
)

print(
    "重复客户数量:",
    test_predictions[
        "SK_ID_CURR"
    ].duplicated().sum()
)

print(
    "TARGET缺失数量:",
    test_predictions[
        "TARGET"
    ].isna().sum()
)


test_predictions.head()

预测结果 Shape: (48744, 2)
重复客户数量: 0
TARGET缺失数量: 0


,SK_ID_CURR,TARGET
0,100001,0.307499
1,100005,0.643616
2,100013,0.245754
3,100028,0.171460
4,100038,0.653619


In [9]:
# ============================================================
# 【保存结果】
# 保存测试集违约概率
# ============================================================

test_predictions.to_csv(
    "data/processed/lightgbm_test_predictions.csv",
    index=False
)

test_predictions.to_parquet(
    "data/processed/lightgbm_test_predictions.parquet",
    index=False
)

print(
    "测试集预测结果保存完成"
)

测试集预测结果保存完成
